# Task 4: General Health Query Chatbot (API Integration)

**Objective**: Create a chatbot that can answer general health-related questions using a secure API key.

### Step 1: Securely Load API Key
We use `python-dotenv` for local execution and a fallback for Kaggle Secrets.

In [1]:
import os
from dotenv import load_dotenv

# Load from .env file (for local use)
load_dotenv()

api_key = os.getenv("GITHUB_TOKEN")

# Fallback for Kaggle Secrets
if not api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        api_key = user_secrets.get_secret("GITHUB_TOKEN")
    except:
        print("API Key not found. Please set GITHUB_TOKEN in your .env file or Kaggle Secrets.")

API Key not found. Please set GITHUB_TOKEN in your .env file or Kaggle Secrets.


### Step 2: Initialize Chatbot Client

In [2]:
# Example using GitHub Models (requires 'openai' library)
from openai import OpenAI

client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=api_key
)

def get_health_response(message, history):
    # Safety Filter
    risk_keywords = ['chest pain', 'suicide', 'self-harm', 'emergency', 'dying']
    if any(k in message.lower() for k in risk_keywords):
        return "⚠️ **Emergency Notice**: This sounds like a medical emergency. Please call local emergency services."

    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": "Act like a helpful medical assistant. Use Markdown for formatting."},
            {"role": "user", "content": message}
        ],
        model="gpt-4o",
        temperature=0.7
    )
    return response.choices[0].message.content

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

### Step 3: Launch Gradio Interface

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=get_health_response,
    title="Health Assistant (Secure API)",
    theme="soft"
)

demo.launch(share=True)